# 第 1 章 統計学の復習: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/01-statistics-review.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0             # 乱数の種（変えると、標本が変わる）
n = 10               # 標本の大きさ
mean = 5.0           # 母集団の平均
sd = 2.0             # 母集団の標準偏差
repetitions = 2000   # 標本を取り直す回数
probability = 0.975  # 臨界値を求める下側確率
degrees_of_freedom = (2, 5, 30)  # 図に描く t 分布の自由度

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 標本の特徴と、標本平均のばらつき

In [ ]:
import numpy as np

from lib.chapter01.statistics import (
    critical_value,
    sample_variance,
    sampling_distribution_of_mean,
    standard_error_of_mean,
)

rng = np.random.default_rng(seed)
x = rng.normal(mean, sd, n)
print(f"標本平均 {x.mean():.3f}、不偏分散 {sample_variance(x):.3f}、"
      f"標準誤差 {standard_error_of_mean(x):.3f}")
print(f"t({n - 1}) の臨界値 {critical_value('t', probability, n - 1):.3f}、"
      f"正規分布の臨界値 {critical_value('normal', probability):.3f}")

means = sampling_distribution_of_mean(n, repetitions, rng)
print(f"標準正規分布から n={n} 個の平均を {repetitions} 回: "
      f"標準偏差 {means.std(ddof=1):.3f}（理論値 1/√n = {1 / np.sqrt(n):.3f}）")

## 図: 正規分布と t 分布

In [ ]:
from lib.chapter01.plot import plot_densities

plot_densities(degrees_of_freedom)

## 試してみよう

- `n` を 30 にすると、標準誤差は 0.301、t の臨界値は 2.045 になり、正規分布の 1.960
  に近づきます。`n` を 3 にすると、t の臨界値は 4.303 に広がります（標本が小さいほど、
  t 分布の裾は厚い）
- 標本平均のばらつき（計算のセルの 3 行目）は、`n` が 10 のとき 0.313、30 のとき
  0.189、3 のとき 0.576 で、どれも理論値 1/√n（0.316・0.183・0.577）に近い値です
- `seed` を 1 にすると、標本が変わり、標本平均は 5.450、標準誤差は 0.420 になります。
  標本が変わると、推定値も変わります